# SageMaker Model Bias Monitor

This notebook adds a SageMaker Clarify Model Bias Monitor to the NYC Motor Vehicle Collisions project.

The existing monitoring notebook already includes Data Quality Monitoring, Model Quality Monitoring, and a CloudWatch dashboard. This notebook adds the missing bias monitoring component.

The model predicts whether a crash results in injury. The target column is `target`, where `1` means an injury occurred and `0` means no injury occurred.

The selected bias facet is `borough`, because prediction behavior may vary across NYC boroughs.

In [1]:
!pip install "sagemaker==2.214.0"

import boto3
import sagemaker
import pandas as pd
import botocore

from sagemaker import get_execution_role
from sagemaker.model_monitor import (
    ModelBiasMonitor,
    CronExpressionGenerator,
    EndpointInput
)
from sagemaker.clarify import (
    BiasConfig,
    DataConfig,
    ModelConfig,
    ModelPredictedLabelConfig
)

sess = sagemaker.Session()
role = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name
sm_client = boto3.client("sagemaker", region_name=region)

print("Region:", region)
print("Bucket:", bucket)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Region: us-east-1
Bucket: sagemaker-us-east-1-016517410335


## Find the Live Endpoint

Notebook 05 creates a live SageMaker endpoint with a name that starts with `collisions-monitor-ep`. This section finds the newest matching endpoint so the bias monitor can attach to it.

In [2]:
endpoint_name = None

endpoints = sm_client.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending"
)

for ep in endpoints["Endpoints"]:
    if ep["EndpointName"].startswith("collisions-monitor-ep"):
        endpoint_name = ep["EndpointName"]
        break

if endpoint_name is None:
    raise ValueError("No endpoint starting with 'collisions-monitor-ep' found. Run notebook 05 first.")

print("Using endpoint:", endpoint_name)

Using endpoint: collisions-monitor-ep-1781585841


## Bias Monitoring Configuration

The bias monitor uses the validation dataset from notebook 04 as the baseline.

Where we use the following:

- Label column: `target`
- Positive label: `1`
- Bias facet: `borough`
- Endpoint: live endpoint created in notebook 05

In [3]:
monitoring_prefix = "nyc-collisions-monitoring"
training_prefix = "aai-540-group6/nyc-collisions-ml"

baseline_data_uri = f"s3://{bucket}/{training_prefix}/validation/val.csv"
bias_output_uri = f"s3://{bucket}/{monitoring_prefix}/model_bias_results"

label_column = "target"
facet_name = "borough"

df_val = pd.read_csv("data_splits/val.csv")
headers = df_val.columns.tolist()

print("Validation shape:", df_val.shape)
print("Baseline S3 path:", baseline_data_uri)
print("Bias output path:", bias_output_uri)
print("Columns:", headers)

if label_column not in df_val.columns:
    raise ValueError(f"Missing label column: {label_column}")

if facet_name not in df_val.columns:
    raise ValueError(f"Missing facet column: {facet_name}")

Validation shape: (4998, 8)
Baseline S3 path: s3://sagemaker-us-east-1-016517410335/aai-540-group6/nyc-collisions-ml/validation/val.csv
Bias output path: s3://sagemaker-us-east-1-016517410335/nyc-collisions-monitoring/model_bias_results
Columns: ['borough', 'month', 'hour', 'is_rush_hour', 'is_weekend', 'cause_category', 'vehicle_type', 'target']


## Create Model Bias Baseline

SageMaker Clarify first creates a baseline from validation data. This baseline is later used to compare future endpoint predictions and detect bias drift.

In [6]:
endpoint_desc = sm_client.describe_endpoint(
    EndpointName=endpoint_name
)

endpoint_config_name = endpoint_desc["EndpointConfigName"]

endpoint_config_desc = sm_client.describe_endpoint_config(
    EndpointConfigName=endpoint_config_name
)

model_name = endpoint_config_desc["ProductionVariants"][0]["ModelName"]

print("Endpoint name:", endpoint_name)
print("Model name:", model_name)

Endpoint name: collisions-monitor-ep-1781585841
Model name: sagemaker-scikit-learn-2026-06-16-04-57-25-460


In [7]:
bias_config = BiasConfig(
    label_values_or_threshold=[1],
    facet_name=facet_name
)

data_config = DataConfig(
    s3_data_input_path=baseline_data_uri,
    s3_output_path=f"{bias_output_uri}/baseline",
    label=label_column,
    headers=headers,
    dataset_type="text/csv"
)

model_config = ModelConfig(
    model_name=model_name,
    instance_type="ml.m5.large",
    instance_count=1,
    accept_type="text/csv",
    content_type="text/csv"
)

predicted_label_config = ModelPredictedLabelConfig(
    label=0
)

model_bias_monitor = ModelBiasMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    max_runtime_in_seconds=1800,
    sagemaker_session=sess
)

model_bias_monitor.suggest_baseline(
    data_config=data_config,
    bias_config=bias_config,
    model_config=model_config,
    model_predicted_label_config=predicted_label_config,
    wait=True
)

print("Model Bias baseline completed.")

INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: 1.0.


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: 1.0.


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


INFO:sagemaker.clarify:Analysis Config: {'dataset_type': 'text/csv', 'headers': ['borough', 'month', 'hour', 'is_rush_hour', 'is_weekend', 'cause_category', 'vehicle_type', 'target'], 'label': 'target', 'label_values_or_threshold': [1], 'facet': [{'name_or_index': 'borough'}], 'methods': {'report': {'name': 'report', 'title': 'Analysis Report'}, 'pre_training_bias': {'methods': 'all'}, 'post_training_bias': {'methods': 'all'}}, 'predictor': {'model_name': 'sagemaker-scikit-learn-2026-06-16-04-57-25-460', 'instance_type': 'ml.m5.large', 'initial_instance_count': 1, 'accept_type': 'text/csv', 'content_type': 'text/csv', 'label': 0}}


...

.

.

.

!

Model Bias baseline completed.


## Create Bias Monitoring Schedule

This section creates an hourly monitoring schedule. The monitor checks captured endpoint predictions and compares them against the bias baseline.

In [10]:
bias_schedule_name = f"{endpoint_name}-bias-monitor"

ground_truth_input = f"s3://{bucket}/{monitoring_prefix}/ground_truth_data"

try:
    sm_client.describe_monitoring_schedule(
        MonitoringScheduleName=bias_schedule_name
    )
    print(f"Schedule already exists: {bias_schedule_name}")

except botocore.exceptions.ClientError:
    model_bias_monitor.create_monitoring_schedule(
        monitor_schedule_name=bias_schedule_name,
        endpoint_input=EndpointInput(
            endpoint_name=endpoint_name,
            destination="/opt/ml/processing/input_data",
            inference_attribute="0"
        ),
        ground_truth_input=ground_truth_input,
        output_s3_uri=f"{bias_output_uri}/schedule",
        constraints=model_bias_monitor.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics=True
    )

    print("Model Bias monitoring schedule created:", bias_schedule_name)

INFO:sagemaker.model_monitor.clarify_model_monitoring:Uploading analysis config to {s3_uri}.


INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: collisions-monitor-ep-1781585841-bias-monitor


Model Bias monitoring schedule created: collisions-monitor-ep-1781585841-bias-monitor
